# Практическая работа № 7. Случайный лес

## Цель работы

Изучить и сравнить модели регрессии на датасете Abalone: дерево решений и случайный лес. Подготовить признаки, настроить гиперпараметры, оценить качество моделей и проанализировать важность признаков.

**Вариант:** уровень B.  
**Датасет:** Abalone (`abalone.data.csv`).  
**Целевая переменная:** `Rings` — количество колец моллюска.

Сначала сравниваются дерево решений и случайный лес из **100 деревьев** (`n_estimators=100`). Затем с помощью перекрёстной проверки подбираются гиперпараметры случайного леса. После обучения анализируется важность признаков и проверяется, как меняется качество модели при удалении наименее важных признаков. Все модели оцениваются на одной и той же тестовой выборке; фактические результаты и выводы следует заполнить после выполнения ноутбука.

Число 100 деревьев выбрано как разумный баланс: усреднение большого числа деревьев снижает разброс предсказаний, а 100 деревьев остаётся померным по времени обучения.

> Если файл данных находится в другой папке, измените `DATA_PATH` в первой кодовой ячейке.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
DATA_PATH = Path("../data/abalone.data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File not found: {DATA_PATH}. Set DATA_PATH to the location of abalone.data.csv."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
print("Missing values by column:")
print(df.isna().sum())


## 1. Подготовка данных и признаков

Признак `gender` кодируется методом one-hot-кодирования, а пропуски в числовых и категориальных признаках заполняются подходящими значениями. Предобработка включена в Pipeline, чтобы при перекрёстной проверке она обучалась только на соответствующих обучающих данных.

In [ ]:
target = "Rings"
X = df.drop(columns=target)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

categorical_features = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ],
    remainder="drop",
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


## 2. Обучение моделей и оценка качества

Данные разделяются на обучающую `X_train` и тестовую `X_test` выборки. Для оценки используются **MAE** (средняя абсолютная ошибка; меньше — лучше), **RMSE** (корень из средней квадратичной ошибки; меньше — лучше) и **R²** (коэффициент детерминации; ближе к 1 — лучше).

In [ ]:
def make_pipeline(regressor, columns=None):
    """Create a preprocessing and estimator pipeline for the selected source columns."""
    if columns is None:
        columns = X.columns.tolist()
    cat = [name for name in categorical_features if name in columns]
    num = [name for name in numeric_features if name in columns]
    prep = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat),
        ],
        remainder="drop",
    )
    return Pipeline([("preprocessor", prep), ("model", regressor)])


def regression_metrics(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        "MAE": mean_absolute_error(y_eval, pred),
        "RMSE": np.sqrt(mean_squared_error(y_eval, pred)),
        "R\u00b2": r2_score(y_eval, pred),
    }


tree = make_pipeline(DecisionTreeRegressor(random_state=RANDOM_STATE))
forest_default = make_pipeline(RandomForestRegressor(
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
))

tree.fit(X_train, y_train)
forest_default.fit(X_train, y_train)

baseline_results = pd.DataFrame(
    {
        "Decision tree": regression_metrics(tree, X_test, y_test),
        "Random forest (100 trees, default)": regression_metrics(forest_default, X_test, y_test),
    }
).T

display(baseline_results.round(3))


## 3. Подбор гиперпараметров леса

Подбираются значения `n_estimators`, `max_depth` и `min_samples_leaf`. Поиск выполняется по метрике MAE с 3-кратной перекрёстной проверкой на обучающей выборке. Для перебора задано 12 комбинаций параметров; затем лучшая модель сравнивается с лесом из 100 деревьев без настройки.

In [ ]:
tuned_pipeline = make_pipeline(RandomForestRegressor(
    random_state=RANDOM_STATE, n_jobs=-1
))

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 15],
    "model__min_samples_leaf": [1, 3, 5],
}

grid_search = GridSearchCV(
    estimator=tuned_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    return_train_score=True,
)
grid_search.fit(X_train, y_train)
forest_tuned = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV MAE:", round(-grid_search.best_score_, 3))

comparison = pd.DataFrame({
    "Random forest (default, 100 trees)": regression_metrics(forest_default, X_test, y_test),
    "Random forest (tuned)": regression_metrics(forest_tuned, X_test, y_test),
}).T
print("Trees in tuned model:", forest_tuned.named_steps["model"].n_estimators)
display(comparison.round(3))


## 4. Важность признаков

`feature_importances_` показывает, насколько каждый признак важен для работы модели. Для категориального `gender` важности one-hot-признаков суммируются по исходному столбцу. Это **не коэффициенты регрессии**; impurity-based важность может быть смещена в пользу признаков с большим числом возможных значений.

In [ ]:
fitted_preprocessor = forest_tuned.named_steps["preprocessor"]
forest = forest_tuned.named_steps["model"]
encoded_names = fitted_preprocessor.get_feature_names_out()
encoded_importances = forest.feature_importances_

importance_encoded = pd.Series(encoded_importances, index=encoded_names)
importance_by_source = {}
for feature in X.columns:
    if feature in numeric_features:
        key = f"num__{feature}"
        importance_by_source[feature] = float(importance_encoded.get(key, 0.0))
    else:
        prefix = f"cat__{feature}_"
        importance_by_source[feature] = float(
            importance_encoded[[name.startswith(prefix) for name in importance_encoded.index]].sum()
        )

feature_importance = pd.Series(importance_by_source, name="Importance").sort_values(ascending=False)
display(feature_importance.to_frame().round(4))

ax = feature_importance.sort_values().plot.barh(figsize=(9, 5), color="steelblue")
ax.set_xlabel("Summed impurity-based importance")
ax.set_ylabel("Source feature")
ax.set_title("Random forest feature importance")
plt.tight_layout()
plt.show()

print("Two most important features:", ", ".join(feature_importance.head(2).index))


## 5. Абляция: удаление наименее важных признаков

Признаки удаляются по возрастанию их важности. Для каждого варианта с одним, двумя или тремя удалёнными признаками модель обучается заново и оценивается на той же тестовой выборке. Это позволяет проверить, изменяется ли качество после исключения наименее важных признаков. При интерпретации учитывайте, что важность рассчитана для настроенной модели.

In [ ]:
least_important_first = feature_importance.sort_values().index.tolist()
selected_params = grid_search.best_params_

ablation_rows = [{
    "Number of removed features": 0,
    "Removed features": "None",
    **regression_metrics(forest_tuned, X_test, y_test),
}]

for n_removed in range(1, min(3, len(least_important_first)) + 1):
    removed = least_important_first[:n_removed]
    kept = [column for column in X.columns if column not in removed]
    regressor = RandomForestRegressor(
        n_estimators=selected_params["model__n_estimators"],
        max_depth=selected_params["model__max_depth"],
        min_samples_leaf=selected_params["model__min_samples_leaf"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    reduced_model = make_pipeline(regressor, columns=kept)
    reduced_model.fit(X_train[kept], y_train)
    ablation_rows.append({
        "Number of removed features": n_removed,
        "Removed features": ", ".join(removed),
        **regression_metrics(reduced_model, X_test[kept], y_test),
    })

ablation_results = pd.DataFrame(ablation_rows).set_index("Number of removed features")
display(ablation_results.round(3))


## 6. Выводы

После выполнения работы заполните выводы и подставьте фактические значения метрик:

1. Дерево решений показало MAE **___**, RMSE **___**, R² **___**; случайный лес из 100 деревьев — MAE **___**, RMSE **___**, R² **___**. Таким образом, случайный лес [лучше / не лучше] дерева по MAE на тестовой выборке.
2. При кросс-валидации выбраны параметры: `n_estimators=___`, `max_depth=___`, `min_samples_leaf=___`. Настроенный лес [лучше / хуже / так же] справился с задачей по сравнению с базовым.
3. Наиболее важными признаками оказались **___** и **___**. Удаление признаков с наименьшей важностью [изменило / не изменило] качество модели.
4. После удаления признаков MAE изменился с **___** до **___**; значение R² изменилось с **___** на **___**. По результатам абляции можно заключить, что **___**.

Для воспроизводимости использовался параметр `random_state=42`. Итоговые выводы следует делать по значениям метрик, полученным при выполнении ноутбука.